# Build a Speech-to-Speech Study Agent — Student Notebook
**Guided workshop · 35 minutes (30–40) · basic Python and ML knowledge · Google Colab**

**Goal:** add a capability to a voice agent and explain how it works.

This is a **cascaded, turn-based** system that runs entirely in Colab with open-source models:
1. Record a short request in Colab, or upload a recording.
2. **ASR** (faster-whisper) converts the recording into text.
3. **The agent** (Qwen3-0.6B) chooses Python tools; Python runs them and returns the results.
4. The agent writes a short reply.
5. **TTS** (MMS-TTS Indonesian) turns that reply into audio; press Play to listen.

Example: **“Tambahkan tugas belajar agentic AI, lalu tampilkan daftarnya.”**

These are three separate models, not one audio-to-audio model, and this is not a live conversation: you record one request at a time.

**Before you start:** open this notebook in Google Colab (File → Upload notebook), then choose **Runtime → Change runtime type → T4 GPU**. You need a microphone (or a 5–15 second WAV/MP3/M4A recording) and headphones or speakers. No API key is needed. Run the cells in order with Shift+Enter.

**Timing: 35 minutes (adaptable to 30–40).** Setup happens together at the start of class. The first run downloads about 2 GB of model weights, which usually takes a few minutes.

| Minutes | Activity |
|---|---|
| 0–3 | Get into groups, open the notebook, and run sections 1–2 |
| 3–8 | Record one request and check the transcript |
| 8–15 | Follow the tool schema, dispatcher, and agent trace |
| 15–18 | Hear the reply and compare it with the text |
| 18–30 | Implement and test `count_tasks()` |
| 30–35 | Discuss one result and prepare the group submission |

**30-minute version:** 5 minutes for the tool walkthrough and 9 minutes for the exercise. **40-minute version:** add 5 minutes for one failure-case test.

**Data and licenses:** everything runs in your Colab runtime; nothing is sent to a paid API. Use invented tasks only. The output voice is **AI-generated**. Tasks live in memory, so restarting the runtime clears them. Licenses: faster-whisper and Whisper weights (MIT), Qwen3-0.6B (Apache-2.0), transformers (Apache-2.0), num2words (LGPL). **MMS-TTS is CC-BY-NC-4.0: fine for this class, not for commercial products.**

**Fallbacks:** `INPUT_MODE = "text"` skips the microphone; `SPEAK = False` skips TTS. Without a GPU the agent still runs on CPU, only more slowly.

Complete the TODOs in section 9 (`count_tasks()`, required). `complete_task()` is an optional challenge; solutions are in a separate lecturer notebook.

## Working together
Get together in groups of 3. Explore the notebook together, discuss your results, and make sure everyone can explain the code.

**Group name:** …  
**Members:** …

Take turns running the notebook, inspecting the tool trace, and recording your findings. Use your own copy of this notebook for your submission.

## 1 — Install packages
Run this once at the start of the session. Colab already has PyTorch and transformers; this adds the ASR library and a number-to-words helper for TTS.

In [ ]:
!pip -q install faster-whisper num2words

## 2 — Load the open-source models
This cell downloads and loads the agent model (Qwen3-0.6B) and the Indonesian TTS model (MMS-TTS). The first run downloads about 2 GB; later runs in the same session are fast.

**Checkpoint:** the cell prints `Models ready on cuda`. If it says `cpu`, switch to a T4 GPU runtime (Runtime → Change runtime type) and run sections 1–2 again; CPU works, but each agent reply is much slower.

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, VitsModel

LLM_NAME = "Qwen/Qwen3-0.6B"        # Apache-2.0; small and fast. "Qwen/Qwen3-1.7B" picks tools more reliably but is larger
TTS_NAME = "facebook/mms-tts-ind"   # Indonesian TTS; CC-BY-NC-4.0 (non-commercial use only)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("transformers", transformers.__version__, "| device:", device)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=dtype).to(device).eval()
tts_tokenizer = AutoTokenizer.from_pretrained(TTS_NAME)
tts_model = VitsModel.from_pretrained(TTS_NAME).to(device).eval()
print("Models ready on", device)

## 3 — Record, upload, or type a request
Run the recorder helper once, then the input cell. The input cell defines `get_request()` and calls it once.

**Record (default):** click **Record**, allow the microphone, wait for **Recording… Speak now**, say “Tambahkan tugas belajar agentic AI, lalu tampilkan daftarnya,” then click **Stop**. Aim for 5–15 seconds (auto-stop at 30). The first recording also downloads the Whisper model.

**Fallbacks:** set `INPUT_MODE = "upload"` for one WAV/MP3/M4A file, or `"text"` to use `TEXT_INPUT`. You can also choose per call, e.g. `get_request(mode="text", text="Ada berapa tugas saya?")`.

`LANGUAGE = "id"` forces Indonesian; use `"en"` for English or `None` for automatic detection. If the microphone is blocked, use upload or text mode.

**Checkpoint:** compare the transcript with what you said. If it is wrong, record again or type the request before running the agent. A transcript alone never runs a tool.

Use the recorder's own **Stop** and **Cancel** buttons; the cell's interrupt button does not control the microphone.

In [ ]:
# Browser microphone helper — run once. It does not open the mic until Record is clicked.
from pathlib import Path
from base64 import b64decode
import uuid
from google.colab.output import eval_js
from IPython.display import Audio, display

RECORDER_JS = "(async () => {\n  const maxSeconds = __MAX_SECONDS__;\n  return await new Promise((resolve) => {\n    const panel = document.createElement('div');\n    panel.style.cssText = 'padding:14px;border:1px solid #999;border-radius:8px;font:15px sans-serif';\n    const label = document.createElement('p');\n    label.textContent = 'Click Record, allow microphone access, then speak. Maximum ' + maxSeconds + ' seconds.';\n    const record = document.createElement('button'); record.textContent = 'Record';\n    const stop = document.createElement('button'); stop.textContent = 'Stop'; stop.disabled = true;\n    const cancel = document.createElement('button'); cancel.textContent = 'Cancel';\n    for (const button of [record, stop, cancel]) button.style.cssText = 'margin:4px;padding:8px 18px';\n    panel.append(label, record, stop, cancel); document.body.appendChild(panel);\n    let stream, recorder, stopTimer, done = false;\n    const chunks = [];\n    const closeMic = () => { if (stream) stream.getTracks().forEach(track => track.stop()); };\n    const finish = (result) => {\n      if (done) return;\n      done = true; clearTimeout(stopTimer); clearTimeout(idleTimer);\n      if (recorder && recorder.state !== 'inactive') recorder.stop();\n      closeMic(); record.disabled = stop.disabled = cancel.disabled = true;\n      label.textContent = result.ok ? 'Recording saved. Microphone off.' : result.error;\n      resolve(result);\n    };\n    const idleTimer = setTimeout(() => finish({ok:false, error:'Recorder timed out. Rerun this cell, or use upload/text mode.'}), 120000);\n    cancel.onclick = () => finish({ok:false, error:'Recording cancelled. Microphone off.'});\n    stop.onclick = () => {\n      stop.disabled = true;\n      if (recorder && recorder.state === 'recording') {\n        label.textContent = 'Saving recording…'; recorder.stop();\n      }\n    };\n    record.onclick = async () => {\n      record.disabled = true;\n      try {\n        if (!navigator.mediaDevices?.getUserMedia || typeof MediaRecorder === 'undefined')\n          throw new Error('This browser cannot record here. Use upload mode.');\n        label.textContent = 'Allow microphone access in your browser…';\n        stream = await navigator.mediaDevices.getUserMedia({audio:true, video:false});\n        if (done) { closeMic(); return; }\n        const mime = ['audio/webm;codecs=opus', 'audio/webm', 'audio/mp4', 'audio/ogg;codecs=opus']\n          .find(type => MediaRecorder.isTypeSupported(type));\n        recorder = mime ? new MediaRecorder(stream, {mimeType:mime}) : new MediaRecorder(stream);\n        recorder.ondataavailable = event => { if (event.data.size) chunks.push(event.data); };\n        recorder.onerror = () => finish({ok:false, error:'Audio recording failed. Try upload mode.'});\n        recorder.onstop = async () => {\n          closeMic(); clearTimeout(stopTimer);\n          if (done) return;\n          try {\n            const blob = new Blob(chunks, {type:recorder.mimeType || chunks[0]?.type || 'audio/webm'});\n            if (!blob.size) throw new Error('No audio captured. Record again and speak for a few seconds.');\n            if (blob.size > 10 * 1024 * 1024) throw new Error('Recording is too large. Try a shorter request.');\n            const dataUrl = await new Promise((ok, fail) => {\n              const reader = new FileReader();\n              reader.onload = () => ok(reader.result);\n              reader.onerror = () => fail(new Error('Could not read the recording.'));\n              reader.readAsDataURL(blob);\n            });\n            finish({ok:true, data_url:dataUrl, mime_type:blob.type});\n          } catch (error) { finish({ok:false, error:error.message}); }\n        };\n        recorder.start(); stop.disabled = false;\n        label.textContent = 'Recording… Speak now, then click Stop. Auto-stop after ' + maxSeconds + ' seconds.';\n        stopTimer = setTimeout(() => stop.onclick(), maxSeconds * 1000);\n      } catch (error) {\n        finish({ok:false, error:(error.name || 'Error') + ': ' + error.message + ' Use upload/text mode if needed.'});\n      }\n    };\n  });\n})()\n"

def record_audio(max_seconds=30):
    if not isinstance(max_seconds, int) or not 1 <= max_seconds <= 30:
        raise ValueError("Choose a recording limit from 1 to 30 seconds.")
    result = eval_js(RECORDER_JS.replace("__MAX_SECONDS__", str(max_seconds)))
    if not isinstance(result, dict) or not result.get("ok"):
        reason = result.get("error", "Recording failed.") if isinstance(result, dict) else "No recording returned."
        raise RuntimeError(reason)
    mime = result["mime_type"].split(";")[0].lower()
    suffix = {"audio/webm": ".webm", "audio/mp4": ".m4a", "audio/ogg": ".ogg", "audio/wav": ".wav"}.get(mime)
    if not suffix:
        raise RuntimeError("Unsupported recording format. Use upload mode with WAV/MP3/M4A.")
    data_url = result["data_url"]
    header, encoded = data_url.split(",", 1)
    if not header.startswith("data:audio/") or ";base64" not in header:
        raise RuntimeError("Invalid audio data returned by the browser.")
    audio_bytes = b64decode(encoded, validate=True)
    if not audio_bytes or len(audio_bytes) > 10 * 1024 * 1024:
        raise RuntimeError("Recording is empty or too large. Try a short request.")
    path = Path(f"recording_{uuid.uuid4().hex[:8]}{suffix}")
    path.write_bytes(audio_bytes)
    print("Recording ready. Press Play below if you want to check it.")
    display(Audio(filename=str(path), autoplay=False))
    return str(path)

print("Recorder helper ready. Run the input cell below to record, upload, or type.")


In [ ]:
import subprocess
import time
import numpy as np

INPUT_MODE = "record"   # "record", "upload", or "text"
LANGUAGE = "id"         # "en" for English, None for automatic detection
WHISPER_SIZE = "small"  # "base" is faster; "small" is more accurate for Indonesian
TEXT_INPUT = "Tambahkan tugas belajar agentic AI, lalu tampilkan daftarnya."
whisper = globals().get("whisper")  # keep a loaded model when this cell is rerun; set to None to reload

def transcribe(audio_path):
    global whisper
    start = time.perf_counter()
    if whisper is None:
        from faster_whisper import WhisperModel
        print("Loading the ASR model; the first time downloads its weights…")
        whisper = WhisperModel(WHISPER_SIZE, device="cpu", compute_type="int8")
    # Decode with ffmpeg (16 kHz mono); faster-whisper's own reader breaks on newer PyAV versions.
    pcm = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", audio_path,
                          "-f", "f32le", "-ac", "1", "-ar", "16000", "-"],
                         capture_output=True, check=True).stdout
    segments, _ = whisper.transcribe(np.frombuffer(pcm, np.float32), beam_size=1, language=LANGUAGE)
    text = " ".join(segment.text.strip() for segment in segments).strip()
    print(f"ASR time: {time.perf_counter() - start:.1f}s")
    return text

def get_request(mode=None, text=None):
    """Record, upload, or type one request and return it as text."""
    mode = mode or INPUT_MODE
    if mode == "text":
        request = (text or TEXT_INPUT).strip()
    elif mode == "record":
        print("Opening the recorder: click Record below, speak, then click Stop.")
        request = transcribe(record_audio(max_seconds=30))
    elif mode == "upload":
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one audio file.")
        request = transcribe(next(iter(uploaded)))
    else:
        raise ValueError('mode must be "record", "upload", or "text".')
    if not request:
        raise ValueError("No words detected. Try clearer audio or text mode.")
    return request

transcript = ""  # cleared first, so a failed recording never reuses an old request
transcript = get_request()
print("Transcript:", transcript)
print("Check it before running the agent in section 8.")

## 4 — Python tools and state
Explore the functions before calling the model. `study_tips()` is a fixed template; it does not retrieve personalized advice.

Run `add_task("Belajar Python")` and `show_tasks()` in a new cell if you want to test them directly. Use `tasks.clear()` before the full demo.

In [ ]:
tasks = []

def add_task(task: str) -> str:
    task = task.strip()
    if not task or len(task) > 150:
        return "Task must contain 1–150 characters."
    tasks.append(task)
    return f"Added: {task}"

def show_tasks() -> str:
    return "No tasks yet." if not tasks else "\n".join(
        f"{i}. {task}" for i, task in enumerate(tasks, 1)
    )

def study_tips(topic: str) -> str:
    topic = topic.strip()[:100]
    return (f"Study tips for {topic}: review the basics, "
            "practice with a small project, and explain what you learned.")

print("Tools ready. Tasks start empty.")
# No API needed: try add_task("Belajar Python") and show_tasks() manually.
# Run tasks.clear() before the full demo if you tried these manual calls.

## 5 — Describe tools to the model
The schema tells the model which functions it may request and what arguments each expects. Compare `add_task` in this cell with its Python definition above.

Keep the schema names consistent with `execute_tool()` in the next cell.

In [ ]:
tools = [
    {"type": "function", "function": {
        "name": "add_task", "description": "Add a study task to the list.",
        "parameters": {"type": "object", "properties": {
            "task": {"type": "string", "description": "Short study task"}},
            "required": ["task"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "show_tasks", "description": "Show the current study task list.",
        "parameters": {"type": "object", "properties": {},
                       "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "study_tips", "description": "Give general study tips for a topic.",
        "parameters": {"type": "object", "properties": {
            "topic": {"type": "string", "description": "Topic to study"}},
            "required": ["topic"], "additionalProperties": False}}},
]

## 6 — The agent loop
The model asks for a tool by writing a `<tool_call>` block with a name and JSON arguments. Python parses the block, validates it in `execute_tool()`, runs the tool, and sends the result back to the model as a `tool` message. The model can then choose another tool based on that result.

We allow **two tool rounds**, enough for add + show. The final round gives the model no tools, so it must write its reply from the results it already has. This limit is a teaching choice.

Each request starts with a fresh conversation; only the task list persists.

`execute_tool()` has a TODO for the `count_tasks()` exercise. The existing tools work before you complete it.

In [ ]:
import json
import re

MAX_TOOL_ROUNDS = 2
SHOW_RAW_OUTPUT = False  # True prints exactly what the model wrote (useful for debugging tool calls)
TOOL_CALL = re.compile(r"<tool_call>(.*?)</tool_call>", re.DOTALL)
SYSTEM_PROMPT = (
    "You are a study assistant for Indonesian students. Use the tools to manage the study task list. "
    "When a tool can answer or do what the user asks, call it instead of asking the user. "
    "For a request to add a task and then show the list, call add_task first, then show_tasks. "
    "Ask a short question only when the user wants to add a task but does not say which task. "
    "Only claim actions that a tool result confirms. You cannot send messages, emails, or reminders. "
    "Reply in the user's language in at most two short sentences."
)

def execute_tool(name, arguments):
    try:
        args = json.loads(arguments)
        if not isinstance(args, dict):
            return "Tool error: arguments must be an object."
        if name == "add_task" and set(args) == {"task"} and isinstance(args["task"], str):
            return add_task(args["task"])
        if name == "show_tasks" and not args:
            return show_tasks()
        if name == "study_tips" and set(args) == {"topic"} and isinstance(args["topic"], str):
            if not args["topic"].strip():
                return "Tool error: topic must not be empty."
            return study_tips(args["topic"])
        # TODO 3: add a count_tasks branch here (empty arguments only), then re-run this cell.
        return "Tool error: invalid name or arguments."
    except (ValueError, TypeError):
        return "Tool error: malformed arguments."

def generate(messages, use_tools):
    prompt = llm_tokenizer.apply_chat_template(
        messages, tools=tools if use_tools else None, add_generation_prompt=True,
        tokenize=False, enable_thinking=False)  # Qwen3: answer directly, no hidden reasoning
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=128, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    # Keep <tool_call> tags in the text; cut at Qwen's end-of-turn marker.
    return llm_tokenizer.decode(new_tokens).split("<|im_end|>")[0].strip()

def parse_tool_calls(text):
    """Return (name, JSON arguments) for each <tool_call> block the model wrote."""
    calls = []
    for raw in TOOL_CALL.findall(text):
        try:
            call = json.loads(raw)
            calls.append((call["name"], json.dumps(call.get("arguments", {}))))
        except (ValueError, KeyError, TypeError):
            calls.append(("invalid", "{}"))  # execute_tool reports it as a tool error
    return calls

def ask_agent(user_text):
    if not isinstance(user_text, str) or not user_text.strip():
        raise ValueError("Please provide a non-empty request.")
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text.strip()}]
    for round_num in range(MAX_TOOL_ROUNDS + 1):
        use_tools = round_num < MAX_TOOL_ROUNDS
        text = generate(messages, use_tools)
        if SHOW_RAW_OUTPUT:
            print("Raw model output:", text)
        calls = parse_tool_calls(text) if use_tools else []
        if not calls:
            reply = TOOL_CALL.sub("", text).strip()
            if not reply:
                raise RuntimeError("No final text reply. Inspect show_tasks() before retrying.")
            return reply
        messages.append({"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": {"name": name, "arguments": arguments}}
            for name, arguments in calls]})
        for name, arguments in calls:
            print("Selected tool:", name, "Arguments:", arguments)
            output = execute_tool(name, arguments)
            print("Tool result:", output)
            messages.append({"role": "tool", "content": output})

print("Agent functions ready. No action has run yet.")

## 7 — Text becomes speech
`speak_reply()` receives the **final reply text**, not your original request. MMS-TTS reads letters, not digits, so the function first writes numbers as Indonesian words (2 → “dua”). Then it displays a player; press Play to listen.

**Checkpoint:** the spoken reply should match the text. TTS does not check the task list or fix an agent mistake. Tell listeners the voice is AI-generated.

This function only generates speech. Running it again never runs the task tools again.

In [ ]:
from IPython.display import Audio, display
from num2words import num2words

SPEAK = True

def speak_reply(reply, autoplay=False):
    if not isinstance(reply, str) or not reply.strip():
        raise ValueError("There is no reply to speak.")
    text = re.sub(r"\d+", lambda match: num2words(int(match.group()), lang="id"), reply)
    inputs = tts_tokenizer(text, return_tensors="pt").to(tts_model.device)
    if inputs["input_ids"].shape[1] == 0:
        raise ValueError("The reply has no speakable text.")
    with torch.no_grad():
        waveform = tts_model(**inputs).waveform[0].float().cpu().numpy()
    print("AI-generated voice. Press Play to listen.")
    display(Audio(waveform, rate=tts_model.config.sampling_rate, autoplay=autoplay))
    return waveform

print("TTS function ready. No speech has been generated yet.")

## 8 — One complete speech-to-speech turn
`run_turn(request)` runs the agent on the request you checked in section 3, prints the real task list, and speaks the reply.

Run the reset cell once for a clean demonstration, then run the turn cell once.

Inspect **request → selected tool/arguments → tool result → actual list → reply text → audio**. The wording can vary.

In [ ]:
# Reset only when you deliberately want a fresh demonstration.
tasks.clear()
last_reply = None
print("Task list cleared.")

In [ ]:
def run_turn(request, autoplay=False):
    """One agent turn: run tools, show the real list, then speak the reply."""
    global last_reply
    last_reply = None
    print("Request:", request)
    start = time.perf_counter()
    try:
        last_reply = ask_agent(request)
        print("Assistant:", last_reply)
    except (ValueError, RuntimeError) as error:
        print(f"Agent did not finish: {error}")
        print("Check the task list before retrying; some actions may already have run.")
    print(f"Agent time: {time.perf_counter() - start:.1f}s")
    print("Actual task list:", show_tasks())
    if last_reply and SPEAK:
        start = time.perf_counter()
        try:
            speak_reply(last_reply, autoplay=autoplay)
            print(f"TTS time: {time.perf_counter() - start:.1f}s")
        except (ValueError, RuntimeError) as error:
            print(f"Speech output unavailable ({error}); keep the text reply.")
            print("Retry speak_reply(last_reply) only. Do not repeat the turn just to fix audio.")
    return last_reply

# One request may change tasks. Do not rerun just to replay the voice.
run_turn(transcript)

### Replay or retry speech without repeating an action
Press Play in the existing player to listen again. If speech generation failed, set `RETRY_SPEECH = True` and run the cell below: it calls only TTS, never the task tools.

If `SPEAK = False`, the turn stays text-only. You can still run `speak_reply(last_reply)` later.

In [ ]:
RETRY_SPEECH = False
if RETRY_SPEECH:
    speak_reply(last_reply)

## 8b — End-to-end mode: talk, then hear the answer
The full speech-to-speech experience in one cell: run it, click **Record**, speak, click **Stop**, and the agent's spoken reply plays automatically. Run the cell again for the next request; the task list is kept.

This cell always records, whatever `INPUT_MODE` says. If no recorder appears, check that no other cell is still running (for example an earlier recorder waiting for Stop).

In sections 3 and 8 you checked the transcript before the agent acted. Here the agent acts right away, as a real voice assistant does, so a misheard word can cause a wrong action. When a result surprises you, read the printed request and tool trace.

To type instead of speaking: `voice_turn(mode="text", text="Ada berapa tugas saya?")`. If the browser blocks autoplay, press Play.

In [ ]:
def voice_turn(mode="record", text=None):
    """Talk, then hear the answer: request → ASR → agent + tools → TTS (plays automatically)."""
    return run_turn(get_request(mode, text), autoplay=True)

voice_turn()  # run this cell again for each new request

## 9 — Main coding exercise: add `count_tasks()` (12 min including checks)
**Goal:** ask “Ada berapa tugas saya?” and hear a count backed by the actual list.

Complete all three connections:
1. **Python function:** replace TODO 1 below with a string containing the task count. The function must not modify `tasks`.
2. **Tool schema:** finish TODO 2 below, then enable registration. Use the exact name `count_tasks`, no arguments, and an appropriate description.
3. **Dispatcher:** in step 6, add a branch for `name == "count_tasks"` with empty arguments. Return `count_tasks()`. Re-run step 6 so Python uses your edit.

Run the checks after implementation. Then ask the agent in the **Test with the agent** cell below the checks. Do not clear tasks.

A plausible answer alone is insufficient: confirm the trace shows `Selected tool: count_tasks`.

**Timebox:** 2 min to locate the three changes, 6 min to implement them, 2 min for local checks, and 2 min for one count request. The remaining code is provided; do not rewrite the whole pipeline. If the agent model is not loaded, the local checks still prove your implementation.

In [ ]:
def count_tasks():
    # TODO 1: return the count as a string without modifying tasks.
    raise NotImplementedError("Implement count_tasks first.")

REGISTER_COUNT_TOOL = False  # Set True after completing TODOs 1 and 2.
COUNT_TOOL = {
    "type": "function",
    "function": {
        "name": "",  # TODO 2: exact Python function name
        "description": "",  # TODO 2: explain when the model should use this tool
        "parameters": {
            "type": "object", "properties": {},
            "required": [], "additionalProperties": False,
        },
    },
}
if REGISTER_COUNT_TOOL:
    assert COUNT_TOOL["function"]["name"] == "count_tasks"
    assert COUNT_TOOL["function"]["description"].strip()
    # Replace an earlier registration rather than creating duplicates.
    tools[:] = [t for t in tools if t["function"]["name"] != "count_tasks"]
    tools.append(COUNT_TOOL)
    print("Schema registered. Remember TODO 3 in execute_tool().")

### Check your implementation
Set `RUN_EXERCISE_CHECKS = True` after the three TODOs. These checks need no API calls and restore your task list afterwards.

In [ ]:
RUN_EXERCISE_CHECKS = False
if RUN_EXERCISE_CHECKS:
    original_tasks = tasks.copy()
    try:
        for samples in ([], ["Python"], ["Python", "Statistics"]):
            tasks[:] = samples
            assert count_tasks() == str(len(samples)), "Incorrect direct count."
            assert execute_tool("count_tasks", "{}") == str(len(samples)), "Check dispatcher."
            assert tasks == samples, "Counting must not change the list."
        assert execute_tool("count_tasks", '{"unexpected": 1}').startswith("Tool error")
        registered = [t for t in tools if t["function"]["name"] == "count_tasks"]
        assert len(registered) == 1, "Register exactly one schema."
        print("PASS: empty list, one task, two tasks, no mutation, dispatcher and schema.")
    finally:
        tasks[:] = original_tasks

### Test with the agent
After the checks pass, ask the agent. The trace should show `Selected tool: count_tasks` with arguments `{}`, and the number in the reply should match the actual task list. The reply plays automatically.

In [ ]:
if REGISTER_COUNT_TOOL:
    voice_turn(mode="text", text="Ada berapa tugas saya?")  # or speak it: voice_turn()
else:
    print("Finish TODOs 1–3 and set REGISTER_COUNT_TOOL = True first.")

## Optional challenge — `complete_task(task_id)` (submit it if you try it)
**Start only after your `count_tasks()` checks pass.** `count_tasks()` stays the main exercise. This challenge is optional; if you try it, include your code and check results in the submission, even if it is unfinished (explain where you got stuck).

**Goal:** say “Tandai tugas nomor satu sebagai selesai.” The agent should call `complete_task` with `{"task_id": 1}`, and the list should then show that task as done.

Everything happens in the cell below:
1. **State:** each task becomes a dictionary, `{"task": "Belajar Python", "completed": False}`. The cell converts existing tasks for you.
2. **TODO A — `add_task()`:** store that dictionary instead of a plain string.
3. **TODO B — `show_tasks()`:** show number, status, and text, e.g. `1. [done] Belajar Python` or `2. [pending] Belajar statistik`.
4. **TODO C — `complete_task(task_id)`:** `task_id` is the number shown by `show_tasks()`, starting at 1.
   - Invalid number (0, too large, not a whole number): return a message starting with `Tool error` and change nothing.
   - Already completed: say so and change nothing.
   - Otherwise set `completed` to `True`. **Never delete the task**, so `count_tasks()` still counts every task.
5. **TODO D — schema:** name `complete_task` and a clear description. The `task_id` argument is already defined.
6. **TODO E — dispatcher:** accept exactly `{"task_id": <whole number>}`. In Python `True` is also an `int`, so reject booleans.

Finish the TODOs, set `ENABLE_COMPLETE_CHALLENGE = True`, run the cell, then run the checks. If you rerun sections 4–6 later, rerun this cell too.

**Bonus — TODO F:** `show_pending_tasks()` lists only unfinished tasks and keeps their original numbers (e.g. `2. Belajar statistik`). Set `REGISTER_PENDING_TOOL = True` so the agent can answer “Tugas apa yang belum selesai?”

In [ ]:
# Optional challenge — start after your count_tasks checks pass.
ENABLE_COMPLETE_CHALLENGE = False
REGISTER_PENDING_TOOL = False  # Bonus: set True after TODO F.

COMPLETE_TOOL = {"type": "function", "function": {
    "name": "",         # TODO D: exact Python function name
    "description": "",  # TODO D: when should the model use this tool?
    "parameters": {"type": "object", "properties": {
        "task_id": {"type": "integer", "description": "Task number shown by show_tasks, starting at 1"}},
        "required": ["task_id"], "additionalProperties": False}}}
PENDING_TOOL = {"type": "function", "function": {
    "name": "show_pending_tasks",
    "description": "Show only unfinished study tasks with their original numbers.",
    "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}}

if ENABLE_COMPLETE_CHALLENGE:
    assert COMPLETE_TOOL["function"]["name"] == "complete_task", "TODO D: set the schema name."
    assert COMPLETE_TOOL["function"]["description"].strip(), "TODO D: add a description."

    # State: convert plain-string tasks into dictionaries (safe to rerun).
    tasks[:] = [{"task": t, "completed": False} if isinstance(t, str) else t for t in tasks]

    def add_task(task: str) -> str:
        task = task.strip()
        if not task or len(task) > 150:
            return "Task must contain 1–150 characters."
        # TODO A: append {"task": task, "completed": False} and return a confirmation.
        raise NotImplementedError("TODO A")

    def show_tasks() -> str:
        if not tasks:
            return "No tasks yet."
        # TODO B: one line per task, e.g. "1. [done] Belajar Python".
        raise NotImplementedError("TODO B")

    def complete_task(task_id: int) -> str:
        # TODO C: validate task_id, handle already-completed tasks, set completed=True.
        raise NotImplementedError("TODO C")

    def show_pending_tasks() -> str:
        # TODO F (bonus): unfinished tasks only, keeping their original numbers.
        raise NotImplementedError("TODO F")

    tools[:] = [t for t in tools if t["function"]["name"] not in ("complete_task", "show_pending_tasks")]
    tools.append(COMPLETE_TOOL)
    if REGISTER_PENDING_TOOL:
        tools.append(PENDING_TOOL)

    # Wrap the section 6 dispatcher once; rerunning this cell does not stack wrappers.
    base_dispatcher = getattr(execute_tool, "base_dispatcher", execute_tool)

    def execute_tool(name, arguments):
        try:
            args = json.loads(arguments)
        except (ValueError, TypeError):
            return "Tool error: malformed arguments."
        if name == "complete_task":
            # TODO E: if args is exactly {"task_id": <int, not bool>}, return complete_task(...).
            return "Tool error: complete_task is not connected yet."
        if name == "show_pending_tasks" and REGISTER_PENDING_TOOL:
            return show_pending_tasks() if args == {} else "Tool error: no arguments expected."
        return base_dispatcher(name, arguments)

    execute_tool.base_dispatcher = base_dispatcher
    print("complete_task challenge enabled. Run the checks below.")

### Check `complete_task()` (no model needed)
Set `RUN_COMPLETE_CHECKS = True` after enabling the challenge. The checks restore your task list afterwards. Then test it with the agent in the next cell.

In [ ]:
RUN_COMPLETE_CHECKS = False
if RUN_COMPLETE_CHECKS:
    saved_tasks = [dict(t) if isinstance(t, dict) else t for t in tasks]
    try:
        tasks.clear()
        add_task("Belajar Python")
        add_task("Belajar statistik")
        assert tasks == [{"task": "Belajar Python", "completed": False},
                         {"task": "Belajar statistik", "completed": False}], "TODO A: store dictionaries."
        assert "pending" in show_tasks().splitlines()[0], "TODO B: show the status."
        before = [dict(t) for t in tasks]
        for bad in ('{"task_id": 0}', '{"task_id": 3}', '{"task_id": "1"}', '{"task_id": true}',
                    '{"task_id": 1.5}', '{}', '{"task_id": 1, "extra": 1}'):
            assert execute_tool("complete_task", bad).startswith("Tool error"), f"Should reject {bad}"
            assert tasks == before, f"{bad} must not change the list."
        assert not execute_tool("complete_task", '{"task_id": 1}').startswith("Tool error"), "TODO C/E"
        assert tasks[0]["completed"] and not tasks[1]["completed"]
        assert count_tasks() == "2", "Completed tasks must stay in the list."
        assert "done" in show_tasks().splitlines()[0], "TODO B: show done tasks."
        before = [dict(t) for t in tasks]
        assert "already" in execute_tool("complete_task", '{"task_id": 1}').lower(), "Report already-completed tasks."
        assert tasks == before, "An already-completed task must not change."
        assert [t["function"]["name"] for t in tools].count("complete_task") == 1
        print("PASS: dictionaries, status, invalid numbers, already completed, count unchanged, schema.")
        try:
            pending = show_pending_tasks()
        except NotImplementedError:
            print("Bonus show_pending_tasks() not implemented (optional).")
        else:
            assert pending == "2. Belajar statistik", "Bonus: unfinished tasks only, original numbers."
            print("PASS (bonus): show_pending_tasks keeps original numbers.")
    finally:
        tasks[:] = saved_tasks

### Test `complete_task()` with the agent
After the checks pass, ask the agent. The trace should show `Selected tool: complete_task` with `{"task_id": 1}`, and the actual list should show task 1 as `[done]`. For the bonus, ask “Tugas apa yang belum selesai?” and look for `show_pending_tasks`.

In [ ]:
if ENABLE_COMPLETE_CHALLENGE:
    voice_turn(mode="text", text="Tandai tugas nomor satu sebagai selesai.")  # or speak it: voice_turn()
    # Bonus (after TODO F): voice_turn(mode="text", text="Tugas apa yang belum selesai?")
else:
    print("Finish TODOs A–E and set ENABLE_COMPLETE_CHALLENGE = True first.")

## 10 — Review one result and reflect (5 min)
**Core session:** use the add + show demo and the count test already completed; compare expected and actual results, then discuss one limitation. You do not need to run every case below.

**Optional test menu:** with 40 minutes, choose one additional case; otherwise explore these after class. Keep notes for the experiment you choose. For typed experiments, use `voice_turn(mode="text", text="your request")`.

| Experiment | Expected observation | Check separately |
|---|---|---|
| Add, then show the list | `add_task`, then `show_tasks`; one new task | tool order and actual list |
| “Ada berapa tugas saya?” | `count_tasks`; correct count; list unchanged | schema/dispatcher/function |
| “Tambahkan itu” | asks for details; list unchanged | ambiguous request handling |
| “Kirim email ke dosen” | explains unsupported action | no false success |
| Unclear recording | transcript may be wrong | compare intended words with transcript |
| TTS unavailable | text reply remains; do not rerun the action | playback failure versus action failure |

Model behavior can vary. Record failures instead of assuming that a fluent response means success. A user may intentionally repeat a task, so this demo does not silently deduplicate requests.

**Discussion:** Which error belongs to ASR, tool selection, Python execution, or TTS? Why does adding speech make an incorrect confirmation feel more convincing?

**Optional challenge (submit it if you try it):** `complete_task(task_id)`, above.

**Self-study only (not submitted):** the date challenge below.

**Self-study:** rewrite the fixed `study_tips()` template. This is smaller than adding a new tool.

## Optional challenge — “What are my tasks for Thursday?”
**Self-study only · not submitted · no class time.** Explore it after the workshop if you are curious.

Use a **fresh runtime** for this challenge (Runtime → Restart session, then run sections 1–9) and do not combine it with `complete_task()`: both change the task structure in different ways.

Try these voice requests:
- “Tambahkan belajar statistik untuk Kamis.”
- “Tambahkan latihan Python untuk Jumat.”
- “Apa saja tugas saya hari Kamis?” / “What are my tasks for Thursday?”

**Expected behavior:** return only Thursday's tasks and say which exact date was used. This stores a one-time due date; it does not create a calendar event, recurring schedule, or reminder.

### What to build
1. Change each task from a string to a dictionary: `{"task": "Statistics revision", "due_date": "2026-11-05"}`. Store dates as `YYYY-MM-DD`; use `None` for unscheduled tasks. Preserve existing tasks by migrating them without inventing dates.
2. Extend `add_task(task, day=None)` and its schema to accept an optional date phrase. Update `show_tasks()` to display the task and due date. `count_tasks()` must still count all entries.
3. Add `resolve_day(day, today=None)` for date validation/resolution and `tasks_for_day(day)` for filtering. Register the query tool and connect both dated operations to the dispatcher.
4. Let the model pass the user's weekday/date phrase to Python; let Python calculate and filter dates. Make the tool description state the rule and make the reply include the resolved date.

### Date rules for this exercise
- A bare weekday (English or Indonesian, e.g. Thursday/Kamis) means the nearest occurrence **including today**, in a configurable classroom timezone (default `Asia/Jakarta`).
- Also accept exact `YYYY-MM-DD` dates and today/hari ini or tomorrow/besok. Preserve explicit past dates; the user may be asking about an earlier task.
- For ambiguous wording such as “next Thursday,” “Kamis depan,” or `05/11`, ask for an exact date rather than guessing.
- Validate the calendar date. An invalid date must not create a task. A query with no matches should say so clearly.

**Example test anchor:** if today is Wednesday, 4 November 2026, Thursday resolves to `2026-11-05`. If today is already Thursday, it resolves to today. On Friday it resolves to the following Thursday. This example is fixed test data, not the current clock.

### Suggested implementation order
First make exact dates and filtering work. Then add weekday resolution and the English/Indonesian names. Finally test through the agent and TTS. Hint: Python's `datetime`, `timedelta`, and `zoneinfo` can handle the date arithmetic; keep `DATE_ANCHOR = None` for live use and set it only for deterministic checks.

This extension changes the task structure. Finish the core checks first. After enabling it, avoid rerunning the baseline function/schema cells; reapply your date changes if you do. Returning to baseline by running sections 4–6 resets the list, so save any test evidence first.

In [ ]:
# Optional workspace: implement your date extension here.
# Keep DATE_ANCHOR = None for live use.
# TODO: task dictionaries, date resolution, dated add/show, filtering, schemas, and dispatcher.
# Run the optional checks only after finishing the implementation.


### Optional date checks (no API)
Set `RUN_DATE_CHECKS = True` only after the implementation is ready. The checks temporarily fix “today,” restore the task list/clock afterwards, and cover weekday boundaries, filtering, invalid dates, and the original count behavior. Errors before implementation are expected.

After the checks, use the real clock (`DATE_ANCHOR = None`) and record or type one dated add request, then a date query. Confirm `tasks_for_day` appears in the tool trace and the reply includes the date. The list alone does not prove the agent called the filter.

In [ ]:
# Optional local checks — run only after implementing/enabling the date challenge.
RUN_DATE_CHECKS = False
if RUN_DATE_CHECKS:
    from datetime import date
    saved_tasks, saved_anchor = tasks.copy(), DATE_ANCHOR
    try:
        DATE_ANCHOR = date(2026, 11, 4)  # Wednesday; deterministic test date.
        assert resolve_day("Thursday") == "2026-11-05"
        assert resolve_day("Kamis") == "2026-11-05"
        assert resolve_day("Thursday", today=date(2026, 11, 5)) == "2026-11-05"
        assert resolve_day("Thursday", today=date(2026, 11, 6)) == "2026-11-12"
        assert resolve_day("Thursday", today=date(2026, 12, 31)) == "2026-12-31"
        assert resolve_day("Thursday", today=date(2027, 1, 1)) == "2027-01-07"
        tasks.clear()
        execute_tool("add_task", '{"task":"Statistics revision","day":"Kamis"}')
        execute_tool("add_task", '{"task":"Python practice","day":"Friday"}')
        execute_tool("add_task", '{"task":"Read notes"}')
        before = tasks.copy()
        result = execute_tool("tasks_for_day", '{"day":"Thursday"}')
        assert "2026-11-05" in result and "Statistics revision" in result
        assert "Python practice" not in result and "Read notes" not in result
        assert tasks == before, "Query must not change tasks."
        assert "No tasks scheduled for 2026-11-07" in tasks_for_day("Saturday")
        assert count_tasks() == "3", "Count must still include all tasks."
        for bad in ("2026-02-30", "next Thursday", "05/11", ""):
            assert add_task("Bad date test", bad).startswith("Tool error")
            assert tasks == before, "Invalid dates must not add a task."
        assert execute_tool("tasks_for_day", '{"day":3}').startswith("Tool error")
        assert execute_tool("tasks_for_day", '{"day":"Kamis","extra":1}').startswith("Tool error")
        names = [tool["function"]["name"] for tool in tools]
        assert names.count("add_task") == names.count("tasks_for_day") == 1
        print("PASS: weekday resolution, exact-date filtering, no matches, validation, no mutation, and counting.")
    finally:
        tasks[:] = saved_tasks
        DATE_ANCHOR = saved_anchor


## Troubleshooting
- **Microphone:** click Record and allow browser access; wait for the recording status before speaking. Use Stop/Cancel in the output to release the mic. If permission is denied, use `INPUT_MODE = "upload"` or `"text"`.
- **Audio:** upload exactly one short WAV/MP3/M4A in upload mode. Choose the correct `LANGUAGE`. For better accuracy, change the Whisper model from `"base"` to `"small"` (slower).
- **Models:** use a T4 GPU runtime. If loading fails with an out-of-memory error, use Runtime → Restart session and run from section 1. The first load downloads about 2 GB.
- **Agent:** a small model sometimes picks the wrong tool or argument. Check the trace and the actual list; record the failure instead of hiding it. Some actions may run before an error, so inspect `show_tasks()` before retrying.
- **No sound:** press Play, check the volume, and read the text reply. Retry only `speak_reply(last_reply)`.
- **Counting:** a function alone is not enough. Check its schema name and dispatcher branch, and re-run the changed cells. Rerunning section 5 replaces `tools`, so rerun section 9 afterwards. Check with `print([t["function"]["name"] for t in tools])`.
- **Agent ignores a tool:** set `SHOW_RAW_OUTPUT = True` in section 6 and rerun it to see exactly what the model wrote. No `<tool_call>` means the model chose not to use a tool; a clearer tool description or the larger `Qwen/Qwen3-1.7B` usually helps.
- **State:** rerunning section 4 or the reset cell clears tasks. A runtime restart clears all variables.

## Group reflection and submission
Submit **one response per group** through the Google Form shared by the lecturer. Include a notebook link the lecturer can open.

| Part | Submit? | Include |
|---|---|---|
| `count_tasks()` exercise | **Required** | Your three TODOs, the PASS output of the checks, and one count request with its tool trace. |
| `complete_task()` challenge | If you tried it | Your code and the check output. Unfinished is fine; explain where you got stuck. |
| Date challenge | No | Self-study only. |

Use the final 5 minutes to agree on your main finding and prepare the link. Finish the written answers after class if needed.

Keep at least one test input and its outputs in your submitted copy. Replace the prompts below with your group's answers; 2–4 sentences per reflection is enough.

### 1. How does your speech-to-speech agent work?
Explain the roles of ASR, the agent, Python tools, and TTS in your own words.

*Your answer: …*

### 2. What tool did your group add or modify? (`count_tasks`, and `complete_task` if you tried it)
Explain what it does and what you changed so the agent could use it.

*Your answer: …*

### 3. What happened when you tested your agent?
| Test input | Expected result | Actual result / evidence |
|---|---|---|
| … | … | … |

Did it work as expected? Compare the tool trace and actual task list with the final reply.

*Your answer: …*

### 4. What is one limitation you noticed, and how would you improve it?
*Your answer: …*

## References and validation
- [faster-whisper](https://github.com/SYSTRAN/faster-whisper) (MIT)
- [Qwen3-0.6B model card](https://huggingface.co/Qwen/Qwen3-0.6B) (Apache-2.0)
- [MMS-TTS Indonesian model card](https://huggingface.co/facebook/mms-tts-ind) (CC-BY-NC-4.0)
- [Transformers: tool use with chat templates](https://huggingface.co/docs/transformers/main/en/chat_extras)
- Browser recording: [Colab JavaScript outputs](https://colab.research.google.com/notebooks/snippets/advanced_outputs.ipynb), [getUserMedia](https://developer.mozilla.org/en-US/docs/Web/API/MediaDevices/getUserMedia), [MediaRecorder](https://developer.mozilla.org/en-US/docs/Web/API/MediaRecorder)

Validation: the Python logic (tool-call parsing, dispatcher, agent loop, exercise and challenge checks) was tested with simulated model outputs. The real models, microphone capture, and playback still need a live Colab run before class.